# Examples and proofs of the work shown in the "Vacuum FEM Discretization" section of my dissertation work

$(L + B) \vec{u} = \frac{1}{k} C \vec{u} \\$
$L_{a, a'} = \int_{\Omega} D(\vec{x}) (\nabla \Lambda_{a}(\vec{x}) \cdot \nabla \Lambda_{a'}(\vec{x})) dV + \int_{d \Omega} \frac{1}{2} \Lambda_{a}(\vec{x}) \Lambda_{a'}(\vec{x}) ds \\$
$B_{a, a'} = \int_\Omega \Sigma_a(\vec{x})\Lambda_{a}(\vec{x}) \Lambda_{a'}(\vec{x}) \; d \Omega \\$
$C_{a, a'} = \int_\Omega \nu\Sigma_f(\vec{x})\Lambda_{a}(\vec{x}) \Lambda_{a'}(\vec{x}) \; d \Omega$

$\Lambda_a(\vec{x})$ is a multidimensional basis function for the vacuum boundary condition (so there are half-hat basis functions defined at the domain boundaries) with flattened index, $a$. 

There are $N = 2^L + 1$ 1D basis functions in each dimension

We separate the diffusion matrix, $L$, into the volume integral and the surface integral term

$L = S_D + R_v$

with

$S_D = \sum_{i} S_{ii}$ where $i$ is a dimension ($x$, $y$, or $z$)

$S_{ij} = \begin{bmatrix}
    \int_{\Omega} D(\vec{x}) \frac{\partial \Lambda_0(\vec{x})}{\partial i} \frac{\partial \Lambda_0(\vec{x})}{\partial j} dV & \int_{\Omega} D(\vec{x}) \frac{\partial \Lambda_0(\vec{x})}{\partial i} \frac{\partial \Lambda_1(\vec{x})}{\partial j} dV & \ldots \\
    \int_{\Omega} D(\vec{x}) \frac{\partial \Lambda_1(\vec{x})}{\partial i} \frac{\partial \Lambda_0(\vec{x})}{\partial j} dV & \int_{\Omega} D(\vec{x}) \frac{\partial \Lambda_1(\vec{x})}{\partial i} \frac{\partial \Lambda_1(\vx)}{\partial j} dV & \ldots \\
    \vdots & \vdots & \ddots
    \end{bmatrix}$

$ R_v = \frac{1}{2}\begin{bmatrix}
        \int_{d\Omega} \Lambda_0(\vec{x}) \Lambda_0(\vec{x}) ds & \int_{d\Omega} \Lambda_0(\vec{x}) \Lambda_1(\vec{x}) ds & \ldots \\
        \int_{d\Omega} \Lambda_1(\vec{x}) \Lambda_0(\vec{x}) ds & \int_{d\Omega} \Lambda_1(\vec{x}) \Lambda_1(\vec{x}) ds & \ldots \\
        \vdots & \vdots & \ddots
    \end{bmatrix} $

$S_{ij}$ and $R_v$ are $N^d \times N^d$ matrices


# Goals of this Notebook:
- Show that we can efficiently block-encode the BPX preconditioned stiffness matrix $F^T S_D F$
- Show that we can efficiently block-encode the BPX preconditioned surface integral matrix $F^T R_v F$
- Show that the condition numbers/singular values of these matrices (and their sum) allows us to apply the pseudoinverse of their sum efficiently

In [2]:
import sys
import math
import numpy as np
import os
sys.path.append(os.getcwd())
import scipy as sp
import scipy.sparse as spsp
from scipy.sparse import csr_matrix, coo_matrix
import itertools
from fast_inversion.BPX import FEM_BPX_helpers as FEM

First we show that $S_{ij}$ can be decomposed as $S_{ij} = C_{L,i}^T (D_A \otimes I_{2^d}) C_{L,j}$ (proven mathematicall in the writeup)

where

$D_A = \begin{bmatrix}
    D_0 & 0 & \ldots \\
    0 & D_1 & \ldots \\
    \vdots & \vdots & \ddots
    \end{bmatrix}$

$D_A$ is a $2^{dL} \times 2^{dL}$ matrix or equivalently $(N-1)^{d} \times (N-1)^{d}$

$C_{L,j} = \left( \bigotimes_{i'=0}^{i-1} \left[ R_{L,1D} \right] \otimes C_{L,1D} \otimes \bigotimes_{i'=i+2}^{d} \left[ R_{L,1D} \right] \right)$
- $\quad C_{L,j}$ is a $2^{d(L+1)} \times (2^L + 1)^d$ matrix

and

$R_{L,1D} = 2^{-L/2} \begin{bmatrix}
        \frac{1}{2} & \frac{1}{2} & 0 & \cdots & 0  \\
        -\frac{1}{2 \sqrt{3}} & \frac{1}{2 \sqrt{3}} & 0 & \cdots & 0 \\
        0 & \frac{1}{2} & \frac{1}{2} & \cdots & 0 \\
        0 & -\frac{1}{2 \sqrt{3}} & \frac{1}{2 \sqrt{3}} & \cdots & 0 \\
        0 & 0 & \frac{1}{2} & \cdots & 0 \\
        0 & 0 & -\frac{1}{2 \sqrt{3}} & \cdots & 0 \\
        \vdots & \vdots & \vdots & \ddots & 0 \\
        0 & 0 & 0 & \cdots & \frac{1}{2} \\
        0 & 0 & 0 & \cdots & \frac{1}{2 \sqrt{3}} \\
    \end{bmatrix} = 2^{-L/2}(I_{2^L} \otimes \begin{bmatrix}
        \frac{1}{2} &  \frac{1}{2} \\
        -\frac{1}{2 \sqrt{3}} &  \frac{1}{2 \sqrt{3}} \\
    \end{bmatrix}) M_2 $
- $\quad R_{L,1D}$ is a $2^{L+1} \times 2^L + 1$ matrix

$C_{L,1D} = 2^{L/2}\begin{bmatrix}
        -1 & 1 & 0 & \cdots & 0 & 0  \\
        0 & 0 & 0 & \cdots & 0 & 0  \\
        0 & -1 & 1 & \cdots & 0 & 0  \\
        0 & 0 & 0 & \cdots & 0 & 0  \\
        0 & 0 & -1 & \cdots & 0 & 0  \\
        \vdots & \vdots & \vdots & \ddots & 0 & 0  \\
        0 & 0 & 0 & 0 & 1 & 0  \\
        0 & 0 & 0 & 0 & 0 & 0  \\
        0 & 0 & 0 & 0 & -1 & 1  \\
        0 & 0 & 0 & 0 & 0 & 0  \\
    \end{bmatrix} = 2^{L/2}(I_{2^L} \otimes \begin{bmatrix}
        -1 &  1 \\
        0 &  0 \\
    \end{bmatrix}) M_2$
- $C_{L,1D}$ is a $2^{L+1} \times 2^L + 1$ matrix

$M_2 = \begin{bmatrix}
        1 & 0 & 0 & \cdots & 0 & 0 \\
        0 & 1 & 0 & \cdots & 0 & 0 \\
        0 & 1 & 0 & \cdots & 0 & 0 \\
        0 & 0 & 1 & \cdots & 0 & 0 \\
        0 & 0 & 1 & \cdots & 0 & 0 \\
        \vdots & \vdots & \vdots & \ddots & 0 & 0 \\
        0 & 0 & 0 & \cdots & 1 & 0 \\
        0 & 0 & 0 & \cdots & 1 & 0 \\
        0 & 0 & 0 & \cdots & 0 & 1 \\
    \end{bmatrix}$

First we make functions to create the $R_{L,1D}$ and $C_{L,1D}$ matrices (took these from FEM_BPX_helpers.py) (keep in mind that these are slightly different from the Dirichlet BC matrices defined in the Deiml paper)

In [4]:
def get_C_l_1D_v(l):
    N = 2 ** l
    M1 = np.kron(np.eye(N), np.array([[-1,1],[0,0]])) # slightly modified from the Deiml paper (switched the 1 and -1), I don't think it majorly affects anything but aligns with my math better

    M2 = np.zeros((2**(l+1), 2**l + 1))
    # Modified M2 for vacuum BC, now an operation performing |i> -> 1/sqrt(2) (|2i> + |2i-1>) for 0 < i < 2**l and |0> -> 1/sqrt(2) |0> and |2**l> -> 1/sqrt(2)|2**(l+1)-1>
    for col in range(1,2**l):
        M2[2*col, col] = 1
        M2[2*col-1, col] = 1
    M2[0, 0] = 1
    M2[2**(l+1)-1, 2**l] = 1
    return 2**(l/2) * M1 @ M2

def get_R_l_1D_v(l):
    N = 2 ** l
    M1 = np.kron(np.eye(N), np.array([[1/2,1/2],[-1/(2*math.sqrt(3)),1/(2*math.sqrt(3))]])) # modified from the Deiml paper, but I think this is correct (Deiml paper might just have a different ordering of basis functions?)

    M2 = np.zeros((2**(l+1), 2**l + 1))
    # Modified M2 for vacuum BC, now an operation performing |i> -> 1/sqrt(2) (|2i> + |2i-1>) for 0 < i < 2**l and |0> -> 1/sqrt(2) |0> and |2**l> -> 1/sqrt(2)|2**(l+1)-1>
    for col in range(1,2**l):
        M2[2*col, col] = 1
        M2[2*col-1, col] = 1
    M2[0, 0] = 1
    M2[2**(l+1)-1, 2**l] = 1

    return 2**(-l/2) * M1 @ M2

and here are examples of $C_{L,1D}$ and $R_{L,1D}$

In [9]:
print(r'$C_{L,1D}$ with l=2:')
print(get_C_l_1D_v(2))

print(r'$R_{L,1D}$ with l=2:')
print(get_R_l_1D_v(2))

$C_{L,1D}$ with l=2:
[[-2.  2.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0. -2.  2.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0. -2.  2.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0. -2.  2.]
 [ 0.  0.  0.  0.  0.]]
$R_{L,1D}$ with l=2:
[[ 0.25        0.25        0.          0.          0.        ]
 [-0.14433757  0.14433757  0.          0.          0.        ]
 [ 0.          0.25        0.25        0.          0.        ]
 [ 0.         -0.14433757  0.14433757  0.          0.        ]
 [ 0.          0.          0.25        0.25        0.        ]
 [ 0.          0.         -0.14433757  0.14433757  0.        ]
 [ 0.          0.          0.          0.25        0.25      ]
 [ 0.          0.          0.         -0.14433757  0.14433757]]


TODO: show how to block-encode the $C_{L,1D}$ and $R_{L,1D}$ matrices explicitly

Now we show the function to create the $C_{L,j}$ matrix from the implementations of the $R_{L,1D}$ and $C_{L,1D}$ matrices

In [29]:
def getC_l_v_deriv(D, D_p, l):
    C_l = np.array([1])
    for _ in range(1,D_p):
        C_l = np.kron(C_l, FEM.get_R_l_1D_v(l))
    C_l = np.kron(C_l, FEM.get_C_l_1D_v(l))
    for _ in range(D_p+1, D+1):
        C_l = np.kron(C_l, FEM.get_R_l_1D_v(l))
    return C_l

Example of the $C_{L,j}$ and $S_{x,x}$ matrices

In [34]:
d = 1
l = 2

print("d=1,    l=2")
C_lx = getC_l_v_deriv(d,1,l)
print(r'C_{L,x}:')
print(C_lx)

D_A = np.eye(int(2**(d*l)))
Sxx = np.transpose(C_lx) @ (np.kron(D_A, np.eye(int(2**d)))) @ C_lx
print(r'S_{x,x}:')
print(Sxx)


d = 2
l = 2
print("\n\nd=2,    l=2, derivative in x direction")
C_lx = getC_l_v_deriv(d,1,l)
print(r'C_{L,x}:')
print(C_lx)

D_A = np.eye(int(2**(d*l)))
Sxx = np.transpose(C_lx) @ (np.kron(D_A, np.eye(int(2**d)))) @ C_lx
print(r'S_{x,x}:')
print(Sxx)


print("\n\nd=2,    l=2, derivative in y direction")
C_lx = getC_l_v_deriv(d,2,l)
print(r'C_{L,x}:')
print(C_lx)

D_A = np.eye(int(2**(d*l)))
Syy = np.transpose(C_lx) @ (np.kron(D_A, np.eye(int(2**d)))) @ C_lx
print(r'S_{x,x}:')
print(Syy)

print("\n\nd=2,    l=2, S_D matrix (full laplacian)")
print(r'S_D:')
S_D = Sxx + Syy
print(S_D)

d=1,    l=2
C_{L,x}:
[[-2.  2.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0. -2.  2.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0. -2.  2.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0. -2.  2.]
 [ 0.  0.  0.  0.  0.]]
S_{x,x}:
[[ 4. -4.  0.  0.  0.]
 [-4.  8. -4.  0.  0.]
 [ 0. -4.  8. -4.  0.]
 [ 0.  0. -4.  8. -4.]
 [ 0.  0.  0. -4.  4.]]


d=2,    l=2, derivative in x direction
C_{L,x}:
[[-0.5        -0.5        -0.         ...  0.          0.
   0.        ]
 [ 0.28867513 -0.28867513 -0.         ...  0.          0.
   0.        ]
 [-0.         -0.5        -0.5        ...  0.          0.
   0.        ]
 ...
 [ 0.          0.         -0.         ... -0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.          0.
   0.        ]
 [ 0.          0.          0.         ...  0.         -0.
   0.        ]]
S_{x,x}:
[[ 0.33333333  0.16666667  0.          0.          0.         -0.33333333
  -0.16666667  0.          0.          0.          0.          0.
   0.          0.         

We can also decopmose the surface integral matrix, $R_v$ using a similar structure (proven mathmatically in the writeup)

$R_v = \sum_i R_{v,i}$

$R_{v,i} = C_{L,R,i}^T C_{L,R,i}$

$C_{L,R,i} = \left( \bigotimes_{i'=0}^{i-1} \left[ R_{L,1D} \right] \otimes B_{L,1D} \otimes \bigotimes_{i'=i+2}^{d} \left[ R_{L,1D} \right] \right)$

$B_{L,1D} = \frac{1}{\sqrt{2}}\begin{bmatrix}
        1 & 0 & 0 & \cdots & 0 & 0  \\
        0 & 0 & 0 & \cdots & 0 & 0  \\
        0 & 0 & 0 & \cdots & 0 & 0  \\
        0 & 0 & 0 & \cdots & 0 & 0  \\
        0 & 0 & 0 & \cdots & 0 & 0  \\
        \vdots & \vdots & \vdots & \ddots & 0 & 0  \\
        0 & 0 & 0 & 0 & 0 & 0  \\
        0 & 0 & 0 & 0 & 0 & 0  \\
        0 & 0 & 0 & 0 & 0 & 1  \\
        0 & 0 & 0 & 0 & 0 & 0  \\
    \end{bmatrix}$

In [ ]:
# should be a 2^(l+1) x 2^l + 1 matrix
def get_B_l_1D_v(l):
    B_L_1D = np.zeros((2**(l+1), 2**l + 1))
    B_L_1D[0,0] = 1/math.sqrt(2) # input of the leftmost half-hat function outputs the constant function in the leftmost region (in the gradient basis)
    B_L_1D[2**(l+1)-2, 2**l] = 1/math.sqrt(2) # input of the rightmost half-hat function outputs the constant function in the rightmost region (in the gradient basis)
    return B_L_1D

In [36]:
def getC_l_r(D, l):
    pi_l_C_l_r = csr_matrix((D*2**(D*(l+1)), (2**l + 1)**D), dtype=float)
    for s in range(1,D+1):
        pi_l_C_l_s = np.array([1])
        for _ in range(1,s):
            pi_l_C_l_s = np.kron(pi_l_C_l_s, get_R_l_1D_v(l))

        pi_l_C_l_s = np.kron(pi_l_C_l_s, get_B_l_1D_v(l))

        for _ in range(s+1, D+1):
            pi_l_C_l_s = np.kron(pi_l_C_l_s, get_R_l_1D_v(l))

        rows, cols = np.nonzero(pi_l_C_l_s)
        pi_l_C_l_s_data = pi_l_C_l_s[rows, cols]

        rows_g = rows + (s-1)*2**(D*(l+1))
        cols_g = cols

        update = coo_matrix((pi_l_C_l_s_data, (rows_g, cols_g)), shape=pi_l_C_l_r.shape)
        pi_l_C_l_r += update.tocsr() 
        
        #pi_l_C_l[(s-1)*2**(D*(l+1)):s*2**(D*(l+1)), :] = pi_l_C_l_s
    pi_l_star_new = FEM.jk_interleave_permutation_matrix(l, D, sparse=False)
    pi_l_new = np.array([pi_l_star_new + 2**(D*l+D) * d for d in range(D)]).flatten()
    C_l_r = pi_l_C_l_r[pi_l_new, :]
    return C_l_r

Examples of the $C_{L,R,i}$ and $R_{v,i}$ matrices

In [ ]:
d = 1
l = 2

print("d=1,    l=2")
C_lrx = getC_l_r(d,l)
print(r'C_{L,x}:')
print(C_lx)
# TODO: change this to output examples of the R_v matrix
'''
D_A = np.eye(int(2**(d*l)))
Sxx = np.transpose(C_lx) @ (np.kron(D_A, np.eye(int(2**d)))) @ C_lx
print(r'S_{x,x}:')
print(Sxx)


d = 2
l = 2
print("\n\nd=2,    l=2, derivative in x direction")
C_lx = getC_l_v_deriv(d,1,l)
print(r'C_{L,x}:')
print(C_lx)

D_A = np.eye(int(2**(d*l)))
Sxx = np.transpose(C_lx) @ (np.kron(D_A, np.eye(int(2**d)))) @ C_lx
print(r'S_{x,x}:')
print(Sxx)


print("\n\nd=2,    l=2, derivative in y direction")
C_lx = getC_l_v_deriv(d,2,l)
print(r'C_{L,x}:')
print(C_lx)

D_A = np.eye(int(2**(d*l)))
Syy = np.transpose(C_lx) @ (np.kron(D_A, np.eye(int(2**d)))) @ C_lx
print(r'S_{x,x}:')
print(Syy)

print("\n\nd=2,    l=2, S_D matrix (full laplacian)")
print(r'S_D:')
S_D = Sxx + Syy
print(S_D)'''